In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from braincoder.models import LogGaussianPRF

In [ ]:
# ── basis parameters (matching fit_aprf_weighted_cv.py defaults) ─────────────
value_min, value_max = 2.5, 41.5
n_basis = 8

modes = np.linspace(value_min, value_max, n_basis).astype(np.float32)
spacing = modes[1] - modes[0]
fwhm = float(2.0 * spacing)   # default: 2× inter-basis spacing

basis_pars = pd.DataFrame({
    'mode':      modes,
    'fwhm':      np.full(n_basis, fwhm, dtype=np.float32),
    'amplitude': np.ones(n_basis,  dtype=np.float32),
    'baseline':  np.zeros(n_basis, dtype=np.float32),
})

print(f'spacing = {spacing:.2f} CHF,  fwhm = {fwhm:.2f} CHF')
print(basis_pars)

In [ ]:
# ── evaluate basis predictions over a fine CHF grid ─────────────────────────
model = LogGaussianPRF(parameterisation='mode_fwhm_natural')

x_fine = np.linspace(0.5, 55.0, 500).astype(np.float32)
paradigm_fine = pd.DataFrame({'x': x_fine})

# basis_predictions returns (n_timepoints, n_basis)
basis_pred = model.basis_predictions(paradigm_fine, basis_pars).numpy()
print(f'basis_pred shape: {basis_pred.shape}')

In [ ]:
# ── colour palette ───────────────────────────────────────────────────────────
colors = cm.plasma(np.linspace(0.1, 0.9, n_basis))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, xvals, xlabel, xscale in [
    (axes[0], x_fine,      'CHF (linear)',  'linear'),
    (axes[1], x_fine,      'CHF (log)',     'log'),
]:
    for k in range(n_basis):
        ax.plot(xvals, basis_pred[:, k], color=colors[k],
                label=f'{modes[k]:.1f} CHF')
    ax.axvline(value_min, color='k', lw=0.8, linestyle=':', alpha=0.5)
    ax.axvline(value_max, color='k', lw=0.8, linestyle=':', alpha=0.5)
    ax.set_xscale(xscale)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Response (a.u.)')
    ax.set_title(f'Log-Gaussian basis pRFs  (n={n_basis}, fwhm={fwhm:.1f} CHF)\n'
                 f'x-axis: {xscale}')
    if xscale == 'log':
        from matplotlib.ticker import ScalarFormatter
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.set_xticks([1, 2, 5, 10, 20, 50])

axes[0].legend(title='mode', fontsize=7, ncol=2, loc='upper right')
plt.suptitle('Basis functions are symmetric in log-CHF space\n'
             'but right-skewed in linear space (narrow at low values, wide at high)',
             fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── compare fwhm choices ─────────────────────────────────────────────────────
fwhm_choices = [spacing * 0.5, spacing * 1.0, spacing * 2.0, spacing * 3.0]

fig, axes = plt.subplots(1, len(fwhm_choices), figsize=(14, 3.5), sharey=True)

for ax, fw in zip(axes, fwhm_choices):
    bp = basis_pars.copy()
    bp['fwhm'] = float(fw)
    bp_pred = model.basis_predictions(paradigm_fine, bp).numpy()
    for k in range(n_basis):
        ax.plot(x_fine, bp_pred[:, k], color=colors[k])
    ax.axvline(value_min, color='k', lw=0.8, linestyle=':', alpha=0.5)
    ax.axvline(value_max, color='k', lw=0.8, linestyle=':', alpha=0.5)
    ax.set_xlabel('CHF')
    ax.set_title(f'fwhm = {fw:.1f} CHF\n({fw/spacing:.1f}× spacing)')

axes[0].set_ylabel('Response (a.u.)')
plt.suptitle(f'Effect of fwhm on basis overlap  (n={n_basis} bases, spacing={spacing:.1f} CHF)',
             fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── example weighted combinations ────────────────────────────────────────────
# Show what different weight vectors look like as a tuning curve

example_weights = {
    'peaked low (5 CHF)':   np.exp(-0.5 * ((modes - 5.0)  / 4.0) ** 2),
    'peaked mid (22 CHF)':  np.exp(-0.5 * ((modes - 22.0) / 4.0) ** 2),
    'peaked high (38 CHF)': np.exp(-0.5 * ((modes - 38.0) / 4.0) ** 2),
    'broad positive':       np.ones(n_basis),
    'low > high':           np.linspace(1, 0, n_basis),
    'high > low':           np.linspace(0, 1, n_basis),
}

# Use default fwhm basis
n_ex = len(example_weights)
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flat

for ax, (label, w) in zip(axes, example_weights.items()):
    w = np.asarray(w, dtype=np.float32)
    tuning = basis_pred @ w          # (n_fine,)

    # stacked basis (faded)
    for k in range(n_basis):
        ax.fill_between(x_fine, basis_pred[:, k] * w[k],
                        alpha=0.25, color=colors[k])
    ax.plot(x_fine, tuning, 'k', lw=2, label='weighted sum')
    ax.axvline(value_min, color='gray', lw=0.8, linestyle=':')
    ax.axvline(value_max, color='gray', lw=0.8, linestyle=':')

    ax2 = ax.inset_axes([0.65, 0.55, 0.32, 0.38])
    ax2.bar(range(n_basis), w, color=colors)
    ax2.set_xticks([])
    ax2.set_yticks([])
    ax2.set_title('weights', fontsize=7)

    ax.set_xlabel('CHF')
    ax.set_ylabel('Response')
    ax.set_title(label, fontsize=9)

plt.suptitle('Example weighted combinations of log-Gaussian basis pRFs', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── compare: equally-spaced modes in linear vs log CHF space ─────────────────
modes_lin = np.linspace(value_min, value_max, n_basis).astype(np.float32)
modes_log = np.exp(np.linspace(np.log(value_min), np.log(value_max), n_basis)).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, modes_use, title in [
    (axes[0], modes_lin, 'Modes equally spaced in linear CHF\n(current default)'),
    (axes[1], modes_log, 'Modes equally spaced in log CHF\n(alternative)'),
]:
    spacing_use = modes_use[1] - modes_use[0] if len(modes_use) > 1 else value_max - value_min
    bp_use = basis_pars.copy()
    bp_use['mode'] = modes_use
    bp_use['fwhm'] = float(2.0 * spacing_use)
    pred_use = model.basis_predictions(paradigm_fine, bp_use).numpy()
    for k in range(n_basis):
        ax.plot(x_fine, pred_use[:, k], color=colors[k], label=f'{modes_use[k]:.1f}')
    ax.axvline(value_min, color='k', lw=0.8, linestyle=':', alpha=0.5)
    ax.axvline(value_max, color='k', lw=0.8, linestyle=':', alpha=0.5)
    ax.set_xlabel('CHF')
    ax.set_ylabel('Response')
    ax.set_title(title)
    ax.legend(title='mode', fontsize=7, ncol=2)

plt.suptitle('Linear vs log mode spacing — coverage of the CHF range', fontsize=10)
plt.tight_layout()
plt.show()